In [1]:
"""
SarcasmLens – Enhanced Hybrid System (Novelty 1 + Novelty 3)
------------------------------------------------------------
Feature Enhancements:
1. Hybrid TF-IDF = Word TF-IDF + Char TF-IDF
2. Code-Mixing Ratio Feature (Hindi/English token ratio)

Models implemented:
1. Logistic Regression
2. Linear SVM
3. RBF SVM
4. Random Forest

This version is the strongest enhancement for Subtask 3.
"""

# =======================================================
# Imports
# =======================================================
import pandas as pd
import numpy as np
import re
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier

In [2]:

# =======================================================
# 1. Load Dataset
# =======================================================
path = r"C:\MAIN\Projects\Sarcasm Detection\Dataset\unique_tweets.csv"
df = pd.read_csv(path)
print(f"Dataset shape: {df.shape}")
print(df.head())
# Auto-detect columns
possible_text_cols = [c for c in df.columns if "tweet" in c.lower() or "text" in c.lower()]
possible_label_cols = [c for c in df.columns if "label" in c.lower()]

text_col = possible_text_cols[0]
label_col = possible_label_cols[0]

df = df[[text_col, label_col]]
df.columns = ["text", "label"]

Dataset shape: (11367, 3)
        ID                                              Tweet Label
0   7640.0  takeout burrito shielded from cold as though i...   YES
1  11848.0  sight of coworkers' stupid fucking faces endur...   YES
2  13098.0                                porch ceded to bats   YES
3   7530.0  panicked donald trump jr. tries to cover up co...   YES
4   7973.0  mike gravel can't believe his polling numbers ...   YES


In [3]:
# =======================================================
# 2. Text Cleaning
# =======================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@[A-Za-z0-9_]+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\u0900-\u097F!?'\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

In [4]:
# =======================================================
# 3. Code-Mixing Ratio Feature
# =======================================================
def is_hindi(word):
    return bool(re.search(r"[\u0900-\u097F]", word))

def compute_code_mix_ratio(text):
    tokens = text.split()
    if len(tokens) == 0:
        return 0.0
    hindi_words = sum(is_hindi(w) for w in tokens)
    return hindi_words / len(tokens)

class CodeMixFeature(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        ratios = [compute_code_mix_ratio(t) for t in X]
        return np.array(ratios).reshape(-1, 1)

In [5]:
# =======================================================
# 4. Train/Test Split
# =======================================================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)


# =======================================================
# 5. Hybrid TF-IDF Vectorizer (Word + Char)
# =======================================================
word_tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=5000
)


# Combine TF-IDF + Code-Mix
combined_features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf),
    ("codemix", Pipeline([
        ("extract", CodeMixFeature()),
        ("scale", StandardScaler())
    ]))
])

In [6]:
# =======================================================
# 6. Model Definitions
# =======================================================
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
    "LinearSVM": LinearSVC(C=1.0, random_state=42),
    "RBFSVM": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
}


# =======================================================
# 7. Train + Evaluate All Hybrid Models
# =======================================================
best_model_name = None
best_f1 = 0.0
best_pipeline = None
results = []

for name, model in models.items():
    print(f"\n============== Training {name} ==============")
    
    pipeline = Pipeline([
        ("features", combined_features),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    results.append({
        "model": name,
        "accuracy": float(acc),
        "weighted_f1": float(f1)
    })
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_pipeline = pipeline




============== Training LogisticRegression ==============
Accuracy: 0.9675
Weighted F1: 0.9675

Classification Report:
               precision    recall  f1-score   support

          NO     0.9593    0.9633    0.9613       954
         YES     0.9734    0.9705    0.9719      1320

    accuracy                         0.9675      2274
   macro avg     0.9663    0.9669    0.9666      2274
weighted avg     0.9675    0.9675    0.9675      2274

Confusion Matrix:
 [[ 919   35]
 [  39 1281]]

============== Training LinearSVM ==============


c:\Users\Preet\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Accuracy: 0.9710
Weighted F1: 0.9710

Classification Report:
               precision    recall  f1-score   support

          NO     0.9654    0.9654    0.9654       954
         YES     0.9750    0.9750    0.9750      1320

    accuracy                         0.9710      2274
   macro avg     0.9702    0.9702    0.9702      2274
weighted avg     0.9710    0.9710    0.9710      2274

Confusion Matrix:
 [[ 921   33]
 [  33 1287]]

============== Training RBFSVM ==============
Accuracy: 0.9723
Weighted F1: 0.9723

Classification Report:
               precision    recall  f1-score   support

          NO     0.9775    0.9560    0.9666       954
         YES     0.9687    0.9841    0.9763      1320

    accuracy                         0.9723      2274
   macro avg     0.9731    0.9700    0.9715      2274
weighted avg     0.9724    0.9723    0.9723      2274

Confusion Matrix:
 [[ 912   42]
 [  21 1299]]

============== Training RandomForest ==============
Accuracy: 0.9776
Weighted F1: 

In [7]:
# =======================================================
# 8. Saves Best Enhanced Hybrid Model
# =======================================================
print("\n====================================")
print(f"Best Hybrid+CodeMix Model: {best_model_name}")
print(f"Best Weighted F1: {best_f1:.4f}")
print("====================================")

joblib.dump(best_pipeline, "enhanced_hybrid_codemix_best_model.pkl")

with open("enhanced_hybrid_codemix_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved enhanced hybrid + codemix model and results.")



Best Hybrid+CodeMix Model: RandomForest
Best Weighted F1: 0.9775

Saved enhanced hybrid + codemix model and results.
